In [10]:
from pathlib import Path
import os
import sys

ROOT = next(
    (
        folder
        for folder in [Path.cwd(), *Path.cwd().parents]
        if (folder / "src" / "wesad" / "models.py").is_file()
    ),
    None,
)

if ROOT is None:
    raise RuntimeError("Không tìm thấy thư mục dự án.")

os.chdir(ROOT)

src_path = str(ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print("Project:", ROOT)
print("Python:", sys.executable)

Project: c:\Users\Home\Desktop\final_ML\DOAN\SGU26_Group09_Machine_Learning\wesad-stress-detection
Python: c:\Users\Home\Desktop\final_ML\DOAN\SGU26_Group09_Machine_Learning\wesad-stress-detection\.venv\Scripts\python.exe


In [9]:
import pandas as pd
from IPython.display import display

from wesad.data import SUBJECTS, load_feature_table
from wesad.utils import read_config, file_hash


# Lấy đường dẫn và checksum từ cấu hình 

In [5]:
defaults = read_config(ROOT / "configs" / "defaults.yaml")

FEATURES = ROOT / defaults["features"]
SCHEMA = ROOT / defaults["schema"]

print("File dữ liệu:", FEATURES)
print("File schema:", SCHEMA)


File dữ liệu: c:\Users\Home\Desktop\final_ML\DOAN\SGU26_Group09_Machine_Learning\wesad-stress-detection\data\features\g2_physiological68_group15_20261107\all_subjects.csv
File schema: c:\Users\Home\Desktop\final_ML\DOAN\SGU26_Group09_Machine_Learning\wesad-stress-detection\data\features\g2_physiological68_group15_20261107\feature_schema.json


# Kiểm tra hai file có tồn tại.

In [6]:
assert FEATURES.is_file(), f"Không tìm thấy dữ liệu: {FEATURES}"
assert SCHEMA.is_file(), f"Không tìm thấy schema: {SCHEMA}"

# Kiểm tra đúng phiên bản dữ liệu.

In [7]:
actual_sha = file_hash(FEATURES)
assert actual_sha == defaults["feature_sha256"], (
    f"Checksum không khớp. Giá trị hiện tại: {actual_sha}"
)


# Đọc dữ liệu và kiểm tra cấu trúc


In [8]:
df, feature_columns = load_feature_table(
    FEATURES,
    SCHEMA,
    expected_subjects=SUBJECTS,
)

assert len(feature_columns) == 68, "Số đặc trưng khác 68."

print("\nĐã kiểm tra checksum và cấu trúc dữ liệu.")
print("Số cửa sổ:", len(df))
print("Số đặc trưng:", len(feature_columns))
print("Số người tham gia:", df["subject_id"].nunique())
print("Số giá trị thiếu trong các đặc trưng:",
      int(df[feature_columns].isna().sum().sum()))

print("\n5 dòng đầu:")
display(df.head())

print("\nSố cửa sổ theo người và nhãn:")
display(
    pd.crosstab(
        df["subject_id"],
        df["label_original"],
    ).reindex(SUBJECTS)
)


Đã kiểm tra checksum và cấu trúc dữ liệu.
Số cửa sổ: 6189
Số đặc trưng: 68
Số người tham gia: 15
Số giá trị thiếu trong các đặc trưng: 37460

5 dòng đầu:


,subject_id,window_id,start_s,end_s,label_original,label_binary,label_fraction,quality_flag,chest_ecg_hr_mean_bpm,chest_ecg_hr_std_bpm,...,wrist_acc_x_5s_abs_integral_mean,wrist_acc_y_5s_mean_mean,wrist_acc_y_5s_std_mean,wrist_acc_y_5s_abs_integral_mean,wrist_acc_z_5s_mean_mean,wrist_acc_z_5s_std_mean,wrist_acc_z_5s_abs_integral_mean,wrist_acc_magnitude_5s_mean_mean,wrist_acc_magnitude_5s_std_mean,wrist_acc_magnitude_5s_abs_integral_mean
0,S2,S2_213500,305.0,365.0,1,0,0.974214,chest_temp:suspect_contact;wrist_bvp:insuffici...,84.815057,7.982102,...,2.881022,0.061051,0.145758,1.155599,0.590462,0.228952,3.180094,0.994083,0.077596,4.970416
1,S2,S2_217000,310.0,370.0,1,0,1.000000,chest_temp:suspect_contact;wrist_bvp:insuffici...,85.253561,7.835214,...,2.724935,0.079696,0.114213,1.072550,0.619442,0.221434,3.324992,0.993719,0.071203,4.968593
2,S2,S2_220500,315.0,375.0,1,0,1.000000,chest_temp:suspect_contact;wrist_bvp:insuffici...,82.243263,8.680372,...,2.586507,0.122770,0.113965,1.001139,0.642440,0.221584,3.439982,0.994281,0.071527,4.971405
3,S2,S2_224000,320.0,380.0,1,0,1.000000,chest_temp:suspect_contact;wrist_bvp:insuffici...,77.477849,9.297856,...,2.443075,0.165519,0.110231,0.930705,0.665535,0.218807,3.555461,0.993828,0.068995,4.969140
4,S2,S2_227500,325.0,385.0,1,0,1.000000,chest_temp:suspect_contact;wrist_bvp:insuffici...,76.148216,8.506286,...,2.428955,0.163289,0.102565,0.912720,0.697616,0.201692,3.533000,0.993622,0.066669,4.968109



Số cửa sổ theo người và nhãn:


label_original,1,2,3
subject_id,,,
S2,219,113,63
S3,218,118,65
S4,222,117,65
S5,230,119,65
S6,226,121,65
S7,228,118,65
S8,224,124,64
S9,227,119,65
S10,226,135,65


# EXP-001 — Dummy baseline

Người thực hiện: Thái
ngày 14/11/2026
## Mục tiêu
Xác định mức đối chứng bằng mô hình luôn dự đoán lớp
phổ biến nhất trong tập train.

## Giả thuyết trước khi chạy
Dummy không sử dụng thông tin đặc trưng để phân biệt trạng thái.
Kết quả là mức tham chiếu cho các mô hình học máy tiếp theo.

## Lượt R01 — Kiểm tra trên S2
- Train: 14 người còn lại.
- Test: S2.
- Thực hiện cả multiclass và binary.
- Chưa tuning và chưa cân bằng lớp.
- Độ đo: Macro-F1 và Balanced Accuracy.
- Chưa dùng lượt kiểm tra này để kết luận cho toàn bộ 15 người.

In [12]:
from copy import deepcopy
from wesad.evaluation import run_experiment

EXP_ID = "EXP-001"
RUN_NO = "r01"


CONFIG = deepcopy(defaults)

CONFIG.update({
    "protocol": "notebook_research_new_runs_v1",
    "experiment_id": EXP_ID,
    "run_no": RUN_NO,
    "run_scope": "quick",
    "models": ["dummy"],
    "tasks": ["multiclass", "binary"],
    "balance": False,
    "nested": False,
    "output_dir": "outputs/notebook_research/EXP-001/r01_quick",
    "source_paths": [
        "src/wesad/data.py",
        "src/wesad/evaluation.py",
        "src/wesad/models.py",
        "src/wesad/utils.py",
    ],
})

RUN_DIR = ROOT / CONFIG["output_dir"]

# Xác nhận lại dữ liệu trước khi chạy.
assert file_hash(FEATURES) == CONFIG["feature_sha256"]

result = run_experiment(
    CONFIG,
    resume=True,
    outer_subjects=["S2"],
)

print("Trạng thái:", result["status"])
print("Thư mục kết quả:", RUN_DIR)

multiclass/dummy/S2: SKIP verified
binary/dummy/S2: SKIP verified
Trạng thái: partial
Thư mục kết quả: c:\Users\Home\Desktop\final_ML\DOAN\SGU26_Group09_Machine_Learning\wesad-stress-detection\outputs\notebook_research\EXP-001\r01_quick


In [13]:
metrics = pd.read_csv(RUN_DIR / "metrics_by_subject.csv")

assert len(metrics) == 2, "Cần đủ kết quả multiclass và binary."
assert set(metrics["subject_id"]) == {"S2"}

display(metrics)

,subject_id,task,model,n_test,macro_f1,balanced_accuracy,seconds
0,S2,multiclass,dummy,395,0.237785,0.333333,0.0946
1,S2,binary,dummy,395,0.416544,0.500000,0.0385


## Nhận xét R01

- Đã hoàn thành kiểm tra Dummy trên S2 cho cả hai bài toán.
- Multiclass: Macro-F1 = 0.237785; Balanced Accuracy = 0.333333.
- Binary: Macro-F1 = 0.416544; Balanced Accuracy = 0.500000.
- Dummy chỉ dự đoán lớp phổ biến nhất trong train,
  không nhận diện được Stress trong lượt này.
- Không ghi nhận cảnh báo trong hai fold.
- Đây là kết quả kiểm tra S2, chưa đại diện cho toàn bộ 15 người.

## Quyết định tiếp theo

Giữ nguyên thiết lập mô hình và chạy LOSO đủ 15 người
trong lượt R02